# 03 — Customer Segmentation (RFM Analysis)

RFM is a standard CRM/customer-analytics technique that scores every customer on three dimensions:

- **Recency (R)** — how many days since their last purchase (lower is better)
- **Frequency (F)** — how many distinct invoices they've placed (higher is better)
- **Monetary (M)** — total revenue they've generated (higher is better)

Each dimension is scored 1–5 (quintiles), then combined into 8 actionable customer segments — **Champions, Loyal Customers, Potential Loyalists, New Customers, At Risk, Can't Lose Them, Hibernating, Lost Customers** — each implying a different account-management action.

**Note on sample size:** this dataset has 50 customer accounts, which is small for quintile-based scoring (segments below rely on relative ranking within this specific customer base, not universal thresholds). The methodology scales directly to a larger customer base.

---


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 5)
pd.options.display.float_format = "{:,.2f}".format

DATA_PATH = Path("../data/retail_sales_cleaned.csv")
FIG_DIR = Path("../reports/figures")
MODEL_DIR = Path("../models")
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
df["date"] = pd.to_datetime(df["date"])
print(f"Shape: {df.shape}, {df['customer'].nunique()} unique customers")


## Step 1 — Compute Recency, Frequency, Monetary per Customer

In [ ]:
snapshot_date = df["date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("customer").agg(
    recency=("date", lambda x: (snapshot_date - x.max()).days),
    frequency=("doc_no", "nunique"),
    monetary=("sales_value", "sum"),
).reset_index()

print(f"Snapshot date (day after last transaction): {snapshot_date.date()}")
rfm.describe()


## Step 2 — Score Each Dimension (Quintiles 1-5)

In [ ]:
# Recency: lower days-since-purchase = better = higher score, so we reverse the quintile labels
rfm["R_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# Frequency & Monetary: higher = better. Rank first to break ties safely with qcut.
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm["RFM_score"] = rfm["R_score"].astype(str) + rfm["F_score"].astype(str) + rfm["M_score"].astype(str)
rfm.sort_values("monetary", ascending=False).head(10)


## Step 3 — Assign Customer Segments

In [ ]:
def assign_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    if r >= 3 and f >= 3 and m >= 3:
        return "Loyal Customers"
    if r >= 4 and f <= 2:
        return "New Customers"
    if r >= 3 and f <= 3 and m <= 3:
        return "Potential Loyalists"
    if r <= 2 and f >= 4 and m >= 4:
        return "Can't Lose Them"
    if r <= 2 and f >= 3:
        return "At Risk"
    if r <= 2 and f <= 2 and m <= 2:
        return "Hibernating"
    return "Lost Customers"

rfm["segment"] = rfm.apply(assign_segment, axis=1)

segment_action = {
    "Champions": "Reward and retain — priority service, early access to new products, ask for referrals.",
    "Loyal Customers": "Upsell/cross-sell — engage regularly, they respond well to loyalty perks.",
    "Potential Loyalists": "Nurture — offer incentives to increase purchase frequency.",
    "New Customers": "Onboard well — build the relationship early with good service and follow-up.",
    "At Risk": "Win back — personalized outreach before they're lost, understand what changed.",
    "Can't Lose Them": "Urgent re-engagement — high value but going quiet, priority account-management attention.",
    "Hibernating": "Low-cost reactivation — win-back campaigns, but not high-touch given low historical value.",
    "Lost Customers": "Deprioritize — unlikely to be worth heavy re-acquisition spend, unless strategically important.",
}
rfm["recommended_action"] = rfm["segment"].map(segment_action)

segment_counts = rfm["segment"].value_counts()
print(segment_counts)


## Step 4 — Visualize Segments

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
order = segment_counts.index
sns.barplot(x=segment_counts.values, y=order, ax=ax, orient="h", palette="viridis")
ax.set_title("Customer Count by RFM Segment")
ax.set_xlabel("Number of Customers")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_rfm_segment_counts.png", dpi=120)
plt.show()


In [ ]:
segment_value = rfm.groupby("segment")["monetary"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=segment_value.values, y=segment_value.index, ax=ax, orient="h", palette="magma")
ax.set_title("Total Revenue by RFM Segment")
ax.set_xlabel("Total Revenue")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_rfm_segment_revenue.png", dpi=120)
plt.show()

for seg, rev in segment_value.items():
    pct = rev / rfm["monetary"].sum() * 100
    print(f"{seg:22s}: R{rev:>14,.0f}  ({pct:5.1f}% of total revenue)")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
segments = rfm["segment"].unique()
palette = dict(zip(segments, sns.color_palette("tab10", len(segments))))
for seg in segments:
    sub = rfm[rfm["segment"] == seg]
    ax.scatter(sub["frequency"], sub["monetary"], s=sub["recency"].apply(lambda x: max(20, 300 - x)),
               label=seg, alpha=0.7, color=palette[seg])
ax.set_xscale("symlog")
ax.set_yscale("log")
ax.set_xlabel("Frequency (invoices, log scale)")
ax.set_ylabel("Monetary (total revenue, log scale)")
ax.set_title("Customer Segments — Frequency vs Monetary\n(bubble size = recency, bigger = more recent)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_rfm_scatter.png", dpi=120)
plt.show()


## Step 5 — Segment Deep-Dives

In [ ]:
print("=== CHAMPIONS (retain & reward) ===")
print(rfm[rfm["segment"] == "Champions"].sort_values("monetary", ascending=False)[["customer", "recency", "frequency", "monetary"]])

print("\n=== AT RISK / CAN'T LOSE THEM (urgent win-back) ===")
at_risk = rfm[rfm["segment"].isin(["At Risk", "Can't Lose Them"])].sort_values("monetary", ascending=False)
print(at_risk[["customer", "recency", "frequency", "monetary", "segment"]])


In [ ]:
print("=== NEW CUSTOMERS (onboard well) ===")
print(rfm[rfm["segment"] == "New Customers"][["customer", "recency", "frequency", "monetary"]])

print("\n=== HIBERNATING / LOST (low-cost reactivation or deprioritize) ===")
low_value = rfm[rfm["segment"].isin(["Hibernating", "Lost Customers"])].sort_values("recency", ascending=False)
print(low_value[["customer", "recency", "frequency", "monetary", "segment"]].head(15))


## Step 6 — Save the Segmentation

In [ ]:
output_cols = ["customer", "recency", "frequency", "monetary", "R_score", "F_score", "M_score",
               "RFM_score", "segment", "recommended_action"]
rfm[output_cols].sort_values("monetary", ascending=False).to_csv(MODEL_DIR / "customer_rfm_segments.csv", index=False)
print(f"Saved: {MODEL_DIR / 'customer_rfm_segments.csv'}")


## Key Takeaways

In [ ]:
champions_pct = segment_value.get("Champions", 0) / rfm["monetary"].sum() * 100
at_risk_customers = rfm[rfm["segment"].isin(["At Risk", "Can't Lose Them"])]
at_risk_value = at_risk_customers["monetary"].sum() / rfm["monetary"].sum() * 100

print(f"- Champions make up {(rfm['segment']=='Champions').sum()} of {len(rfm)} customers but drive "
      f"{champions_pct:.1f}% of total revenue — the clearest priority for account management.")
print(f"- {len(at_risk_customers)} customers are 'At Risk' or 'Can't Lose Them', representing "
      f"{at_risk_value:.1f}% of total revenue currently going quiet — the highest-value win-back opportunity.")
print(f"- {(rfm['segment']=='Lost Customers').sum()} customers are fully 'Lost' — useful to know for "
      f"realistic pipeline/revenue planning rather than counting on their return.")
print(f"\nNext: 04_sales_forecasting.ipynb for revenue forecasts.")
